# Gen11 Brakes — Combined Thermal Analysis

Unified Jupyter notebook for both operating cases:
- **Case A — 1g Dynamic Braking** (from `1g dynamic.py`) — transient stop $v_0 \to 0$
- **Case B — Continuous Downhill** (from `continuous downhill.py`) — steady-speed grade holding

All calculations are **front total + rear total**. Vehicle has **2 wheels front, 1 wheel rear**.
Ideal distribution assumes braking force split proportional to dynamic normal loads.

Run all cells top-to-bottom after editing the shared **Vehicle Inputs** cell.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

g = 9.81
print("imports ok — g =", g)

: 

## Shared Vehicle Inputs (edit once, used by both cases)

From **Gen 11 Mechanical VDR, 2025-26.docx**: $w=281\,kg, wb=2.30\,m, z_{cog}=47.30\,cm$

In [ ]:
# ── SHARED VEHICLE (from Gen 11 Mechanical VDR, 2025-26.docx) ──
total_mass = 281.0              # kg
static_mass_front = 183.0       # kg
static_mass_rear = 98.0         # kg  (front+rear must = total)
axle_distance = 2.30            # m wheelbase L
cog_height = 0.473              # m CoG height h (47.30 cm)

# ── CASE A: 1g dynamic ──
initial_speed = 30.0   # m/s
final_speed = 0.0      # m/s
decel = 9.81  # or 1g  # m/s^2

# ── CASE B: continuous downhill ──
constant_speed = 25.0        # m/s steady
downhill_angle = 15.0        # degrees
downhill_distance = 2000.0   # m along slope

# derived geometry (shared)
total_weight = total_mass * g
front_to_cog = axle_distance * static_mass_rear / total_mass
rear_to_cog  = axle_distance - front_to_cog
print(f"Vehicle: {total_mass} kg  L={axle_distance}m  h={cog_height:.4f}m  front_to_cog={front_to_cog:.3f}m rear_to_cog={rear_to_cog:.3f}m")
print(f"Case A: {initial_speed}→{final_speed} m/s @ {decel/g:.2f}g")
print(f"Case B: {constant_speed} m/s @ {downhill_angle}° for {downhill_distance/1000:.1f} km")

## Case A — 1g Dynamic Braking

$F_1 = (W\cdot b_{rear} + m a h)/L$

In [ ]:
energy_change = 0.5*total_mass*(initial_speed**2 - final_speed**2)
braking_force_total_A = total_mass * decel
t_stop = (initial_speed-final_speed)/decel
avg_power_A = energy_change / t_stop

N1_A = (total_weight*rear_to_cog + total_mass*decel*cog_height)/axle_distance
N2_A = total_weight - N1_A
B1_A = braking_force_total_A * N1_A/total_weight
B2_A = braking_force_total_A - B1_A

print("--- Braking Force Distribution ---")
print(f"Front: {B1_A:,.0f} N ({B1_A/braking_force_total_A*100:.1f}%)")
print(f"Rear:  {B2_A:,.0f} N ({B2_A/braking_force_total_A*100:.1f}%)")
print(f"Total: {braking_force_total_A:,.0f} N")

print("--- Power (Heating) ---")
print(f"Front: {avg_power_A * B1_A / braking_force_total_A / 1000:.1f} kW")
print(f"Rear:  {avg_power_A * B2_A / braking_force_total_A / 1000:.1f} kW")
print(f"Total: {avg_power_A/1000:.1f} kW")

print("\n--- Per Wheel ---")
print(f"Front (each): {B1_A/2:,.0f} N, {avg_power_A * B1_A / braking_force_total_A / 2 / 1000:.1f} kW")
print(f"Rear (single): {B2_A:,.0f} N, {avg_power_A * B2_A / braking_force_total_A / 1000:.1f} kW")


## Case B — Continuous Downhill (steady speed)

$N_1=(W\cos\theta\cdot b + W\sin\theta\cdot h)/L$

In [ ]:
th = math.radians(downhill_angle)
Wn = total_weight*math.cos(th)
Wp = total_weight*math.sin(th)
Btot_B = Wp
Ptot_B = Btot_B * constant_speed
t_desc = downhill_distance/constant_speed

N1_B = (total_weight*math.cos(th)*rear_to_cog + total_weight*math.sin(th)*cog_height)/axle_distance
N2_B = Wn - N1_B
B1_B = Btot_B * N1_B / Wn
B2_B = Btot_B - B1_B
P1_B = Ptot_B * N1_B / Wn
P2_B = Ptot_B - P1_B

print("--- Braking Force Distribution ---")
print(f"Front: {B1_B:,.0f} N ({B1_B/Btot_B*100:.1f}%)")
print(f"Rear:  {B2_B:,.0f} N ({B2_B/Btot_B*100:.1f}%)")
print(f"Total: {Btot_B:,.0f} N")

print("--- Power (Heating) ---")
print(f"Front: {P1_B/1000:.2f} kW ({P1_B/Ptot_B*100:.1f}%)")
print(f"Rear:  {P2_B/1000:.2f} kW ({P2_B/Ptot_B*100:.1f}%)")
print(f"Total: {Ptot_B/1000:.2f} kW")

print("\n--- Per Wheel ---")
print(f"Front (each): {B1_B/2:,.0f} N, {P1_B/2/1000:.2f} kW")
print(f"Rear (single): {B2_B:,.0f} N, {P2_B/1000:.2f} kW")


## Comparison Plots

In [ ]:
labels = ['Front', 'Rear']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Row 1: Case A
axes[0,0].bar(labels, [B1_A, B2_A], color=['#1f77b4','#ff7f0e'])
axes[0,0].set_title('A — Braking Force (N)')
axes[0,1].bar(labels, [avg_power_A * B1_A / braking_force_total_A / 1000, avg_power_A * B2_A / braking_force_total_A / 1000], color=['#1f77b4','#ff7f0e'])
axes[0,1].set_title('A — Power (kW)')
for ax in axes[0]:
    for c in ax.containers:
        ax.bar_label(c, fmt='%.1f', padding=3, fontsize=8)

# Row 2: Case B
axes[1,0].bar(labels, [B1_B, B2_B], color=['#2ca02c','#d62728'])
axes[1,0].set_title('B — Braking Force (N)')
axes[1,1].bar(labels, [P1_B/1000, P2_B/1000], color=['#2ca02c','#d62728'])
axes[1,1].set_title('B — Power (kW)')
for ax in axes[1]:
    for c in ax.containers:
        ax.bar_label(c, fmt='%.2f', padding=3, fontsize=8)

fig.suptitle('Gen11 Brakes — Case A (1g stop) vs Case B (steady downhill)' , fontsize=12)
plt.tight_layout()
for ax in axes.flat:
    ax.grid(True, alpha=0.3)
plt.show()

print(f"A front bias: {B1_A/braking_force_total_A*100:.1f}%  |  B front bias: {B1_B/Btot_B*100:.1f}%")
print(f"A avg power: {avg_power_A/1000:.1f} kW for {t_stop:.1f}s  |  B continuous: {Ptot_B/1000:.2f} kW for {t_desc/60:.1f} min")
